# PERMANOVA in R

## Overview

PERMANOVA (Permutational Multivariate Analysis of Variance) tests whether groups differ in multivariate composition. It partitions a dissimilarity matrix into among-group and within-group components and assesses significance by permutation — making no distributional assumptions.

| Property | Details |
|---|---|
| Null hypothesis | Group centroids are identical in multivariate space |
| Test statistic | Pseudo-F (ratio of among-group to within-group dissimilarity) |
| Significance | Assessed by permuting sample labels (not parametric distribution) |
| Sensitivity | Sensitive to both centroid differences AND dispersion differences |
| Implementation | `vegan::adonis2()` (preferred; supersedes `adonis()`) |

> **Always pair PERMANOVA with `betadisper()`** — see `multivariate_assumptions.ipynb`. A significant PERMANOVA p-value could reflect different centroids, different dispersions, or both.

## Applications by Sector

| Sector | Example |
|---|---|
| **Ecology** | Do invertebrate communities differ significantly between reference, degraded, and restored sites? Does community composition differ between seasons after controlling for site? |
| **Healthcare** | Does gut microbiome composition differ between disease and control groups after controlling for age and sex? |
| **Finance** | Do portfolio compositions differ significantly across fund categories? |

---

## Assumptions Checklist

- [ ] **Exchangeable samples under H₀:** Samples within groups should be exchangeable — satisfied if samples are independent within groups
- [ ] **Homogeneous dispersion (ideally):** PERMANOVA is most interpretable when within-group dispersion is similar across groups — test with `betadisper()`
- [ ] **Correct permutation scheme for complex designs:** Crossed and nested designs require restricted permutations — use the `permute` package
- [ ] **Sufficient permutations:** 999 minimum; 9999 for publication

---

## Setup

In [1]:
library(tidyverse)
library(vegan)
library(permute)    # restricted permutation schemes
library(pairwiseAdonis)  # pairwise PERMANOVA: install from GitHub if needed
# devtools::install_github("pmartinezarbizu/pairwiseAdonis/pairwiseAdonis")

set.seed(42)

# ── Simulate community matrix (same as multivariate_assumptions.ipynb) ────────
n_sites <- 45; n_sp <- 20
habitat <- rep(c("reference", "degraded", "restored"), each = 15)
season  <- rep(rep(c("spring", "summer", "autumn"), each = 5), 3)

ref_means  <- c(8,7,6,5,5,4,3,3,2,2,1,1,1,0,0,0,0,0,0,0)
deg_means  <- c(0,0,1,1,2,3,4,5,6,7,8,6,4,3,2,1,1,1,0,0)
rest_means <- c(4,4,4,3,3,3,3,2,2,2,2,2,1,1,1,1,0,0,0,0)

sim_comm <- function(n, mu, sd = 0.7)
  t(replicate(n, pmax(round(exp(rnorm(n_sp, log(mu + 0.1), sd))), 0L)))

comm_mat <- rbind(sim_comm(15, ref_means),
                  sim_comm(15, deg_means, sd = 1.0),
                  sim_comm(15, rest_means, sd = 0.8))
rownames(comm_mat) <- paste0("site", 1:n_sites)
colnames(comm_mat) <- paste0("sp",   1:n_sp)

env_df <- tibble(
  site    = paste0("site", 1:n_sites),
  habitat = factor(habitat, levels = c("reference", "degraded", "restored")),
  season  = factor(season),
  pH      = c(rnorm(15, 7.8, 0.3), rnorm(15, 6.5, 0.6), rnorm(15, 7.2, 0.4)),
  nitrate = c(rnorm(15, 0.5, 0.2), rnorm(15, 3.5, 1.0), rnorm(15, 1.5, 0.5))
)

dist_bray <- vegan::vegdist(comm_mat, method = "bray")

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'vegan' was built under R version 4.4.3"
Loading required package: permute

Loading required package: clust

---

## Step 1: Always Check Dispersion First

In [2]:
# ── Betadisper before PERMANOVA ───────────────────────────────────────────────
bd <- vegan::betadisper(dist_bray, group = env_df$habitat)
permutest(bd, permutations = 999)
# If significant: PERMANOVA result conflates centroid and dispersion effects
# → Report both tests; discuss which effect is more plausible

cat("\nGroup dispersions (distance to centroid):\n")
print(round(bd$group.distances, 3))


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 999

Response: Distances
          Df   Sum Sq   Mean Sq      F N.Perm Pr(>F)   
Groups     2 0.061576 0.0307881 5.5569    999   0.01 **
Residuals 42 0.232700 0.0055405                        
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1


Group dispersions (distance to centroid):
reference  degraded  restored 
    0.249     0.336     0.273 


---

## One-Way PERMANOVA

**Question:** Do invertebrate communities differ between habitat types?

In [3]:
# ── vegan::adonis2() — preferred over the deprecated adonis() ────────────────
perm_habitat <- vegan::adonis2(
  dist_bray ~ habitat,
  data         = env_df,
  permutations = 999,
  by           = "margin"   # Type III: each term tested after all others
                             # by = "terms": Type I (sequential) — order matters
)
print(perm_habitat)
# Output columns:
#   Df:    degrees of freedom
#   SumOfSqs: sum of squared distances
#   R2:    proportion of total variance explained (effect size)
#   F:     pseudo-F statistic
#   Pr(>F): permutation p-value

cat(sprintf("\nR² for habitat: %.3f (proportion of compositional variance explained)\n",
            perm_habitat$R2[1]))

Permutation test for adonis under reduced model
Marginal effects of terms
Permutation: free
Number of permutations: 999

vegan::adonis2(formula = dist_bray ~ habitat, data = env_df, permutations = 999, by = "margin")
         Df SumOfSqs     R2      F Pr(>F)    
habitat   2   2.6997 0.4052 14.306  0.001 ***
Residual 42   3.9630 0.5948                  
Total    44   6.6628 1.0000                  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

R² for habitat: 0.405 (proportion of compositional variance explained)


---

## Multi-Factor PERMANOVA

**Question:** Do communities differ by habitat after controlling for season?

In [4]:
# ── Two-factor PERMANOVA ──────────────────────────────────────────────────────
perm_multi <- vegan::adonis2(
  dist_bray ~ habitat + season,
  data         = env_df,
  permutations = 999,
  by           = "margin"   # each term tested marginal to the other
)
print(perm_multi)

# ── With continuous covariates ────────────────────────────────────────────────
perm_covar <- vegan::adonis2(
  dist_bray ~ pH + nitrate + habitat,
  data         = env_df,
  permutations = 999,
  by           = "margin"
)
print(perm_covar)

# ── Interaction term ──────────────────────────────────────────────────────────
perm_interact <- vegan::adonis2(
  dist_bray ~ habitat * season,
  data         = env_df,
  permutations = 999,
  by           = "terms"   # interaction requires sequential (Type I) SS
)
print(perm_interact)

Permutation test for adonis under reduced model
Marginal effects of terms
Permutation: free
Number of permutations: 999

vegan::adonis2(formula = dist_bray ~ habitat + season, data = env_df, permutations = 999, by = "margin")
         Df SumOfSqs      R2       F Pr(>F)    
habitat   2   2.6997 0.40520 14.3397  0.001 ***
season    2   0.1977 0.02967  1.0499  0.371    
Residual 40   3.7654 0.56514                   
Total    44   6.6628 1.00000                   
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Permutation test for adonis under reduced model
Marginal effects of terms
Permutation: free
Number of permutations: 999

vegan::adonis2(formula = dist_bray ~ pH + nitrate + habitat, data = env_df, permutations = 999, by = "margin")
         Df SumOfSqs      R2      F Pr(>F)    
pH        1   0.0655 0.00983 0.6991  0.665    
nitrate   1   0.1563 0.02346 1.6690  0.114    
habitat   2   0.8935 0.13411 4.7713  0.001 ***
Residual 40   3.7455 0.56216                  


---

## Pairwise PERMANOVA

When PERMANOVA is significant with 3+ groups, pairwise tests identify which specific pairs of groups differ. P-values must be corrected for multiple comparisons.

In [5]:
# ── Pairwise PERMANOVA via pairwiseAdonis ─────────────────────────────────────
# install.packages("devtools")
# devtools::install_github("pmartinezarbizu/pairwiseAdonis/pairwiseAdonis")
library(pairwiseAdonis)

pairwise_result <- pairwiseAdonis::pairwise.adonis2(
  dist_bray ~ habitat,
  data         = env_df,
  p.adjust.m   = "BH",      # Benjamini-Hochberg FDR correction
  permutations = 999
)
print(pairwise_result)
# Reports: F, R², raw p-value, adjusted p-value for each pair

# ── Manual pairwise if pairwiseAdonis unavailable ─────────────────────────────
pairs <- combn(levels(env_df$habitat), 2, simplify = FALSE)
pairwise_manual <- map_dfr(pairs, function(p) {
  idx  <- env_df$habitat %in% p
  d_sub <- vegan::vegdist(comm_mat[idx, ], method = "bray")
  g_sub <- env_df$habitat[idx]
  res   <- vegan::adonis2(d_sub ~ g_sub, permutations = 999)
  tibble(comparison = paste(p, collapse = " vs. "),
         R2 = round(res$R2[1], 3),
         F  = round(res$F[1], 3),
         p  = res$`Pr(>F)`[1])
}) %>%
  mutate(p_adj = round(p.adjust(p, method = "BH"), 4))

print(pairwise_manual)

$parent_call
[1] "dist_bray ~ habitat , strata = Null , permutations 999"

$reference_vs_degraded
         Df SumOfSqs      R2      F Pr(>F)    
Model     1   2.2173 0.44364 22.327  0.001 ***
Residual 28   2.7807 0.55636                  
Total    29   4.9980 1.00000                  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

$reference_vs_restored
         Df SumOfSqs      R2      F Pr(>F)    
Model     1  0.38493 0.15157 5.0021  0.001 ***
Residual 28  2.15468 0.84843                  
Total    29  2.53961 1.00000                  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

$degraded_vs_restored
         Df SumOfSqs      R2     F Pr(>F)    
Model     1   1.4473 0.32612 13.55  0.001 ***
Residual 28   2.9907 0.67388                 
Total    29   4.4381 1.00000                 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

attr(,"class")
[1] "pwadstrata" "list"      
# A tibble: 3 × 5
  comparison                R2

---

## Restricted Permutations for Complex Designs

When data are not freely exchangeable (e.g., multiple samples per site, repeated measures, nested designs), use the `permute` package to define a restricted permutation scheme.

In [6]:
# ── Restricted permutations: permute within sites, not across ─────────────────
# Scenario: 3 samples per site, 15 sites — samples within a site are not
# exchangeable with samples from other sites

# Define permutation design
perm_design <- permute::how(
  within = permute::Within(type = "free"),          # permute freely within blocks
  plots  = permute::Plots(strata = env_df$habitat),  # block = habitat
  nperm  = 999
)

perm_restricted <- vegan::adonis2(
  dist_bray ~ habitat,
  data         = env_df,
  permutations = perm_design
)
print(perm_restricted)
# Use restricted permutations whenever samples are not fully exchangeable
# Standard PERMANOVA with unrestricted permutations on non-exchangeable data
# produces anti-conservative p-values

Permutation test for adonis under reduced model
Plots: env_df$habitat, plot permutation: none
Permutation: free
Number of permutations: 999

vegan::adonis2(formula = dist_bray ~ habitat, data = env_df, permutations = perm_design)
         Df SumOfSqs     R2      F Pr(>F)
Model     2   2.6997 0.4052 14.306      1
Residual 42   3.9630 0.5948              
Total    44   6.6628 1.0000              


---

## Reporting Results

In [7]:
# ── Clean results table ───────────────────────────────────────────────────────
as.data.frame(perm_habitat) %>%
  rownames_to_column("Term") %>%
  mutate(across(where(is.numeric), ~ round(.x, 4)))

# Standard reporting format:
# "Invertebrate community composition differed significantly among habitat
#  types (PERMANOVA: F(2,42) = X.XX, R² = X.XX, p = .XXX, 999 permutations).
#  Pairwise comparisons indicated that reference communities differed from
#  both degraded (R² = X.XX, p_adj = .XXX) and restored (R² = X.XX,
#  p_adj = .XXX) sites, while degraded and restored communities did not
#  differ significantly (p_adj = .XXX). Homogeneity of multivariate
#  dispersion was [supported / not supported] (betadisper: F = X.XX,
#  p = .XXX), and results should be interpreted accordingly."
#
# Always report: F statistic, df, R², p, number of permutations
# Always report: betadisper result alongside PERMANOVA

Term,Df,SumOfSqs,R2,F,Pr(>F)
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
habitat,2,2.6997,0.4052,14.3057,0.001
Residual,42,3.9630,0.5948,NA,NA
Total,44,6.6628,1.0000,NA,NA


---

## Common Pitfalls

**1. Using `adonis()` instead of `adonis2()`**  
`adonis()` is deprecated and uses Type I (sequential) SS by default, meaning term order affects results. Use `adonis2()` with `by = "margin"` for Type III tests where term order does not matter.

**2. Not reporting betadisper alongside PERMANOVA**  
PERMANOVA detects both centroid and dispersion differences. Without betadisper, you cannot determine which effect is driving a significant result. Always report both.

**3. Treating R² as equivalent to R² in linear regression**  
PERMANOVA's R² measures the proportion of total dissimilarity explained by the grouping factor. With many groups or correlated predictors, R² values can be inflated. Report it as effect size, not as variance explained in the traditional sense.

**4. Not correcting for multiple comparisons in pairwise tests**  
Running 3 or more pairwise PERMANOVAs without correction inflates family-wise error rate. Always apply Benjamini-Hochberg or Bonferroni correction.

**5. Using unrestricted permutations on non-exchangeable data**  
If samples are not freely interchangeable (repeated measures, nested designs, temporal series), unrestricted permutations produce invalid p-values. Use the `permute` package to define the correct scheme.

**6. Not reporting the number of permutations**  
Always state the number of permutations used. 999 is minimum; 9999 is recommended for publication, especially for p-values near the 0.05 threshold.

---
*r_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*